# boolean-mask-combine composite — cx25: combine per-ray and per-triangle masks across a repeated dim

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `boolean-mask-combine`, `einops-repeat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "boolean-mask-combine"
DD_ATOM_IDS = ["boolean-mask-combine", "einops-repeat"]
DD_SUBTOPICS = ["Numpy: Boolean mask combine", "Einops: Repeat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

In ARENA ray-tracing we end up with **two predicates that live on different axes**:
- `ray_ok` — shape `(NR,)` — "this ray is in-bounds / not degenerate".
- `tri_ok` — shape `(NT,)` — "this triangle is front-facing".

To AND them into a per-(ray, triangle) mask of shape `(NR, NT)` we need to **align the axes**. The pattern: `einops.repeat` the per-ray mask along a new `NT` axis (and the per-triangle mask along a new `NR` axis), then boolean-AND the two `(NR, NT)` masks elementwise.

Why `repeat` and not raw broadcasting? `repeat` makes the intent explicit and the resulting tensors have the matching final shape — convenient when downstream code asserts on `mask.shape`.

**Anatomy.** `repeat(ray_ok, 'r -> r t', t=NT)` materializes the ray mask across triangles; `repeat(tri_ok, 't -> r t', r=NR)` does the symmetric thing. Combine with `&`.

### Composite Exercise — combine per-ray and per-triangle masks across a repeated dim

**Atoms exercised together**: `boolean-mask-combine`, `einops-repeat`

Implement `cx25_combine_ray_tri_masks(ray_ok, tri_ok)`.

- `ray_ok`: boolean tensor of shape `(NR,)`.
- `tri_ok`: boolean tensor of shape `(NT,)`.
- Return: boolean tensor of shape `(NR, NT)` where `out[r, t] == ray_ok[r] & tri_ok[t]`.

1. **Repeat** — use `einops.repeat` to lift `ray_ok` to shape `(NR, NT)` and `tri_ok` to shape `(NR, NT)`.
2. **Boolean combine** — AND the two `(NR, NT)` masks.

The test fuzzes random `(NR, NT)` shapes and cross-checks against the naive outer-AND `ray_ok[:, None] & tri_ok[None, :]`.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx25_combine_ray_tri_masks(ray_ok, tri_ok):
    raise NotImplementedError

def _test_cx25():
    # atom-coverage: enforce the solution uses einops.repeat (not .expand or full tile-copy)
    import inspect as _inspect
    _src = _inspect.getsource(cx25_combine_ray_tri_masks)
    assert 'repeat(' in _src, 'must use einops.repeat'
    # Case A: tiny hand-built example.
    ray_ok = t.tensor([True, False, True, True])
    tri_ok = t.tensor([True, True, False])
    out = cx25_combine_ray_tri_masks(ray_ok, tri_ok)
    assert out.dtype == t.bool, f'expected bool, got {out.dtype}'
    assert tuple(out.shape) == (4, 3), f'expected (4,3), got {tuple(out.shape)}'
    expected = ray_ok[:, None] & tri_ok[None, :]
    assert t.equal(out, expected), 'mask mismatch — did you AND across the right axes?'

    # Case B: edge — all True.
    ray_ok = t.ones(5, dtype=t.bool)
    tri_ok = t.ones(7, dtype=t.bool)
    out = cx25_combine_ray_tri_masks(ray_ok, tri_ok)
    assert tuple(out.shape) == (5, 7)
    assert out.all().item()

    # Case C: edge — one side all False.
    ray_ok = t.zeros(3, dtype=t.bool)
    tri_ok = t.tensor([True, False, True, True])
    out = cx25_combine_ray_tri_masks(ray_ok, tri_ok)
    assert tuple(out.shape) == (3, 4)
    assert not out.any().item()

    # Case D: fuzz against the naive outer-AND reference.
    rng = t.Generator().manual_seed(17)
    for NR, NT in [(2, 3), (10, 4), (1, 8), (8, 1), (16, 16)]:
        ro = (t.rand(NR, generator=rng) > 0.4)
        to_ = (t.rand(NT, generator=rng) > 0.4)
        out = cx25_combine_ray_tri_masks(ro, to_)
        assert tuple(out.shape) == (NR, NT)
        assert t.equal(out, ro[:, None] & to_[None, :])
    _dd_passed.add('cx25')

_test_cx25()

<details><summary>Show solution — cx25</summary>

```python
def cx25_combine_ray_tri_masks(ray_ok, tri_ok):
    NR = ray_ok.shape[0]
    NT = tri_ok.shape[0]
    # Atom A (einops-repeat): lift each per-axis mask onto the joint (NR, NT) grid.
    ray_grid = repeat(ray_ok, 'r -> r t', t=NT)
    tri_grid = repeat(tri_ok, 't -> r t', r=NR)
    # Atom B (boolean-mask-combine): elementwise AND on the matching shape.
    return ray_grid & tri_grid
```

`einops.repeat` here is doing the same job as `ray_ok[:, None].expand(NR, NT)` — the result is logically broadcast across the new axis. The explicit `repeat` makes the joint axis intent self-documenting, which is a win in long ARENA ray-tracing pipelines where many `(NR, NT)` tensors need to line up axis-for-axis.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx25'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx25',
        'subtopics': ["Numpy: Boolean mask combine", "Einops: Repeat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()